In [1]:
import pandas as pd
import json
import mne
import numpy as np
import mne
import os
from itertools import product
import glob

# --------------------------------------------------------------------------
# REPRODUCIBILITY & HARDWARE SETUP (Must be first)
# ---------------------------------------------------------------------------
print("it started")
import os
import random
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

import numpy as np
import tensorflow as tf
import torch

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.config.threading.set_inter_op_parallelism_threads(1)
tf.config.threading.set_intra_op_parallelism_threads(1)

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

# ---------------------------------------------------------------------------
# ORIGINAL IMPORTS & SETUP
# ---------------------------------------------------------------------------
import json
import uuid
import pandas as pd
import matplotlib.pyplot as plt

# Scipy & MNE
import mne
from scipy.signal import stft, welch
from scipy.stats import entropy, norm
from sklearn.model_selection import KFold, train_test_split

# Scikit-learn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.utils.class_weight import compute_class_weight

# TensorFlow / Keras
from tensorflow.keras import layers, models, Model, callbacks

print(f"Reproducibility settings locked with SEED: {SEED}")

# GPU Check
if tf.config.list_physical_devices('GPU'):
    print("TensorFlow GPU Accelerated Backend Active.")
else:
    print("No GPU detected for TensorFlow. Using CPU.")

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"CUDA GPU Accelerated Backend Active: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")

it started
Reproducibility settings locked with SEED: 42
No GPU detected for TensorFlow. Using CPU.


2026-08-28 16:59:18.081159: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [2]:
# Path to the participants TSV file
file_path = "/kaggle/input/datasets/adithyarajnarayanan/eeg-pd-dataset-part-3/EEG dataset part 3 /participants.tsv"

# Read the tab-separated file
df = pd.read_csv(file_path, sep='\t')

# Display the first few rows
df.head()

,participant_id,subject_id,group,updrs_part_iii,updrs_total,moca,age,sex,disease_duration,ledd,pigd_score,td_score,ctt
0,sub-001,HC0001,HC,0.0,0.0,30.0,42.0,M,NaN,NaN,NaN,NaN,NaN
1,sub-002,HC0003,HC,2.0,3.0,27.0,60.0,M,NaN,NaN,NaN,NaN,66.0
2,sub-003,HC0004,HC,0.0,1.0,27.0,60.0,F,NaN,NaN,NaN,NaN,63.0
3,sub-004,HC0005,HC,1.0,1.0,25.0,72.0,M,NaN,NaN,NaN,NaN,116.0
4,sub-005,HC0006,HC,NaN,NaN,NaN,47.0,M,NaN,NaN,NaN,NaN,NaN


In [3]:
sub_condition = df.iloc[:,2].values
print(sub_condition[0:5])

['HC' 'HC' 'HC' 'HC' 'HC']


In [4]:
nan_counts = df.isna().sum()
print(nan_counts)

participant_id       0
subject_id           0
group                0
updrs_part_iii       5
updrs_total          5
moca                 4
age                  0
sex                  0
disease_duration    28
ledd                29
pigd_score          31
td_score            31
ctt                  9
dtype: int64


In [5]:
missing_ids = []

for i in range(1, 145):
    sub_id = f"{i:03d}"
    file_path = f"/kaggle/input/datasets/adithyarajnarayanan/eeg-pd-dataset-part-3/EEG dataset part 3 /sub-{sub_id}/eeg/sub-{sub_id}_task-rest_eeg.set"

    # Check if file exists; if not, store or print i
    if not os.path.exists(file_path):
        print(i)
        missing_ids.append(i)

In [6]:
missing_ids = []

for i in range(1, 145):
    # Format i with 3-digit zero-padding (e.g., 001, 002, ..., 144)
    sub_id = f"{i:03d}"
    file_path = f"/kaggle/input/datasets/adithyarajnarayanan/eeg-pd-dataset-part-3/EEG dataset part 3 /sub-{sub_id}/eeg/sub-{sub_id}_task-walk_eeg.set"

    # Check if the file does NOT exist and print i
    if not os.path.exists(file_path):
        print(i)
        missing_ids.append(i)

1
5
16
20
25
36
43
84
100
120
126


In [7]:


# Format participant ID as 3-digit zero-padded string ('001')
sub_id = f"{1:03d}"
file_path = f"/kaggle/input/datasets/adithyarajnarayanan/eeg-pd-dataset-part-3/EEG dataset part 3 /sub-{sub_id}/eeg/sub-{sub_id}_task-rest_eeg.set"

# Load the file into memory
raw = mne.io.read_raw_eeglab(file_path, preload=True)

# Extract raw numerical array: shape is (Channels, Length)
signal = raw.get_data()

print("Signal shape (C, L):", signal.shape)

Signal shape (C, L): (65, 60964)


/tmp/ipykernel_58/1231990765.py:6: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True)


In [8]:
sfreq = raw.info['sfreq']

print(f"Sampling Frequency: {sfreq} Hz")

Sampling Frequency: 250.0 Hz


In [9]:

def get_eeg_signal(sub_id, task="walk", band="full",target_sfreq=256,duration=2.0, notch_freq=50.0,
    reject_threshold=0.00028,
    base_dir="/kaggle/input/datasets/adithyarajnarayanan/eeg-pd-dataset-part-3/EEG dataset part 3 "
):
    """
    Loads, cleans, filters by frequency band, resamples, and segments EEG data.
    Returns float32 NumPy array of shape (N_epochs, Channels, Time).
    """
    # 1. Map band names to frequency limits
    band_limits = {
        'full':  (1.0, 45.0),
        'delta': (1.0, 4.0),
        'theta': (4.0, 8.0),
        'alpha': (8.0, 12.0),
        'beta':  (12.0, 30.0),
        'gamma': (30.0, 45.0)
    }

    if band.lower() not in band_limits:
        raise ValueError(f"Invalid band '{band}'. Choose from: {list(band_limits.keys())}")

    l_freq, h_freq = band_limits[band.lower()]

    # 2. Format subject ID
    if isinstance(sub_id, int):
        sub_str = f"{sub_id:03d}"
    else:
        sub_str = str(sub_id).zfill(3)

    file_path = f"{base_dir}/sub-{sub_str}/eeg/sub-{sub_str}_task-{task}_eeg.set"

    # 3. Load continuous file
    raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)

    # 4. Fix channel types & isolate EEG
    channel_type_mapping = {
        'EOG1': 'eog', 'EOG2': 'eog', 'EOG3': 'eog', 'EOG4': 'eog', 'VREF': 'misc'
    }
    existing_mapping = {ch: t for ch, t in channel_type_mapping.items() if ch in raw.ch_names}
    if existing_mapping:
        raw.set_channel_types(existing_mapping)

    raw.pick_types(eeg=True, eog=False, misc=False)

    # 5. PREPROCESSING
    # A. Bandpass filter for selected band
    raw.filter(l_freq=l_freq, h_freq=h_freq, fir_design='firwin', verbose=False)

    # B. Notch Filter (only if applicable to selected band range)
    if notch_freq is not None and h_freq >= notch_freq:
        raw.notch_filter(freqs=notch_freq, verbose=False)

    # C. Common Average Reference (CAR)
    raw.set_eeg_reference(ref_channels='average', projection=False, verbose=False)

    # 6. Resample to target frequency (e.g., 128 Hz)
    raw.resample(sfreq=target_sfreq, verbose=False)

    # 7. Epoching with Artifact Rejection
    events = mne.make_fixed_length_events(raw, duration=duration)
    reject_criteria = dict(eeg=reject_threshold) if reject_threshold is not None else None

    epochs = mne.Epochs(
        raw,
        events=events,
        tmin=0,
        tmax=duration - (1 / target_sfreq),  # Exactly 256 samples at 128 Hz
        baseline=None,
        reject=reject_criteria,
        preload=True,
        verbose=False
    )

    # Extract array and cast to float32 to save RAM
    signal = epochs.get_data().astype(np.float32)

    return signal


# --- Example Usage ---
# Extract Beta band for walking task at 128 Hz
beta_walk = get_eeg_signal(sub_id=3, task="walk", band="beta")

# Extract Full spectrum (1-45 Hz) for resting task at 128 Hz
full_rest = get_eeg_signal(sub_id=3, task="rest", band="full")

print("Beta Walk Signal shape (N, C, T):", beta_walk.shape)  # e.g., (N, 60, 256)
print("Full Rest Signal shape (N, C, T):", full_rest.shape)  # e.g., (N, 60, 256)

NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


/tmp/ipykernel_58/3059442721.py:33: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)
/tmp/ipykernel_58/3059442721.py:41: RuntimeWarning: The unit for channel(s) VREF has changed from V to NA.
  raw.set_channel_types(existing_mapping)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


/tmp/ipykernel_58/3059442721.py:33: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)
/tmp/ipykernel_58/3059442721.py:41: RuntimeWarning: The unit for channel(s) VREF has changed from V to NA.
  raw.set_channel_types(existing_mapping)


Beta Walk Signal shape (N, C, T): (121, 60, 512)
Full Rest Signal shape (N, C, T): (121, 60, 512)


In [10]:
non_walk_ids = [1,5,16,20,25,36,43,84,100,120,126]

In [11]:
rest_ids = [i for i in range(1,145)]
walk_ids = [i for i in range(1,145) if i not in non_walk_ids]

In [12]:
def get_data(task, band, rest_ids, walk_ids):
    X_hc = []
    X_pd = []

    # Select subject list based on task
    sub_ids = rest_ids if task == "rest" else walk_ids

    for sub_id in sub_ids:
        try:
            # Extract signal for the current subject
            eeg_signal = get_eeg_signal(sub_id=sub_id, task=task, band=band)

            # Split into HC (< 29) or PD (>= 29)
            if sub_id < 29:
                X_hc.append(eeg_signal)
            else:
                X_pd.append(eeg_signal)

        except Exception as e:
            print(f"Skipping Subject {sub_id} ({task}, {band}) due to error: {e}")

    return X_hc, X_pd


In [13]:
def balance_matrices_subject_wise(X_list_c0, X_list_c1):
    c0_windows_per_sub = [sub.shape[0] for sub in X_list_c0]
    c1_windows_per_sub = [sub.shape[0] for sub in X_list_c1]

    total_c0 = sum(c0_windows_per_sub)
    total_c1 = sum(c1_windows_per_sub)

    if total_c0 == total_c1:
        return np.concatenate(X_list_c0, axis=0), np.concatenate(X_list_c1, axis=0)

    if total_c1 > total_c0:
        maj_list = X_list_c1
        maj_counts = np.array(c1_windows_per_sub)
        target_total = total_c0
        is_c1_majority = True
    else:
        maj_list = X_list_c0
        maj_counts = np.array(c0_windows_per_sub)
        target_total = total_c1
        is_c1_majority = False

    num_maj_subs = len(maj_list)
    allocations = np.zeros(num_maj_subs, dtype=int)
    remaining_target = target_total
    active_subs = np.ones(num_maj_subs, dtype=bool)

    while remaining_target > 0 and np.any(active_subs):
        num_active = np.sum(active_subs)
        base_share = remaining_target // num_active
        remainder = remaining_target % num_active

        if base_share == 0:
            chosen_indices = np.where(active_subs)[0][:remaining_target]
            for idx in chosen_indices:
                allocations[idx] += 1
            break

        for i in range(num_maj_subs):
            if active_subs[i]:
                share = base_share + (1 if remainder > 0 else 0)
                remainder -= 1 if remainder > 0 else 0

                available = maj_counts[i] - allocations[i]
                take = min(share, available)

                allocations[i] += take
                remaining_target -= take

                if allocations[i] == maj_counts[i]:
                    active_subs[i] = False

    processed_maj_list = []
    rng = np.random.default_rng(SEED)
    for i, sub_windows in enumerate(maj_list):
        n_needed = allocations[i]
        if n_needed > 0:
            chosen_indices = rng.choice(sub_windows.shape[0], size=n_needed, replace=False)
            processed_maj_list.append(sub_windows[chosen_indices])

    X_processed_maj = np.concatenate(processed_maj_list, axis=0)

    if is_c1_majority:
        return np.concatenate(X_list_c0, axis=0), X_processed_maj
    else:
        return X_processed_maj, np.concatenate(X_list_c1, axis=0)

In [14]:
def scale_data(X_list):
    scaled = []
    for sub in X_list:
        flat = sub.reshape(-1, sub.shape[-1])
        mu = np.mean(flat, axis=0)
        std = np.std(flat, axis=0) + 1e-8
        scaled.append((sub - mu) / std)
    return scaled

In [15]:
def create_inception_time(
    input_shape=(60, 512),
    nb_classes=1,
    nb_filters=32,
    use_residual=True,
    use_bottleneck=True,
    depth=6,
    kernel_size=40,
    bottleneck_size=32
):
    """
    InceptionTime model adapted for EEG tasks.
    Supports input_shape as:
    - 2D: (Channels, Timepoints) -> Reshaped to 1D Conv (Timepoints, Channels)
    - 3D: (Channels, Timepoints, 1) or (1, Channels, Timepoints) or (Timepoints, Channels)
    """
    inputs = layers.Input(shape=input_shape)

    # Standardize shape to 2D tensor (Timepoints, Channels) for Conv1D compatibility
    if len(input_shape) == 2:
        # (Channels, Timepoints) -> (Timepoints, Channels)
        x = layers.Permute((2, 1))(inputs)
    elif len(input_shape) == 3:
        if input_shape[0] == 1:
            # (1, Channels, Timepoints) -> (Timepoints, Channels)
            x = layers.Permute((3, 2, 1))(inputs)
            x = layers.Reshape((input_shape[2], input_shape[1]))(x)
        elif input_shape[2] == 1:
            # (Channels, Timepoints, 1) -> (Timepoints, Channels)
            x = layers.Permute((2, 1, 3))(inputs)
            x = layers.Reshape((input_shape[1], input_shape[0]))(x)
        else:
            # (Timepoints, Channels, 1) or already (Timepoints, Channels)
            x = inputs
    else:
        raise ValueError(f"Expected input_shape length 2 or 3, received: {input_shape}")

    input_res = x
    kernel_size_s = [max(1, kernel_size // (2 ** i)) for i in range(3)]

    # --- Inception Modules Depth Loop ---
    for d in range(depth):
        # 1. Bottleneck Layer
        if use_bottleneck and int(x.shape[-1]) > 1:
            input_inception = layers.Conv1D(
                filters=bottleneck_size, kernel_size=1,
                padding='same', use_bias=False
            )(x)
        else:
            input_inception = x

        # 2. Multi-kernel Convolutions
        conv_list = []
        for k_size in kernel_size_s:
            conv_list.append(
                layers.Conv1D(
                    filters=nb_filters, kernel_size=k_size,
                    padding='same', use_bias=False
                )(input_inception)
            )

        # 3. Max Pooling Branch
        max_pool_1 = layers.MaxPool1D(pool_size=3, strides=1, padding='same')(x)
        conv_6 = layers.Conv1D(
            filters=nb_filters, kernel_size=1,
            padding='same', use_bias=False
        )(max_pool_1)
        conv_list.append(conv_6)

        # 4. Concatenation & Normalization
        x = layers.Concatenate(axis=-1)(conv_list)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)

        # 5. Residual Shortcut Connection every 3 modules
        if use_residual and d % 3 == 2:
            shortcut_y = layers.Conv1D(
                filters=int(x.shape[-1]), kernel_size=1,
                padding='same', use_bias=False
            )(input_res)
            shortcut_y = layers.BatchNormalization()(shortcut_y)

            x = layers.Add()([shortcut_y, x])
            x = layers.Activation('relu')(x)
            input_res = x

    # --- Classification Head ---
    gap_layer = layers.GlobalAveragePooling1D()(x)
    activation = 'sigmoid' if nb_classes == 1 else 'softmax'
    outputs = layers.Dense(nb_classes, activation=activation)(gap_layer)

    model = models.Model(inputs=inputs, outputs=outputs, name="InceptionTime")
    return model

In [16]:
def format_eeg_tensor_inception(data_array):
    """
    Formats input EEG data array into standard 3D matrix for InceptionTime:
    (Batch/Epochs, Timepoints, Channels).
    """
    arr = np.asarray(data_array, dtype=np.float32)

    if arr.ndim == 2:
        # Single Epoch (Channels, Time) -> Transpose to (Time, Channels) -> (1, Time, Channels)
        return np.expand_dims(arr.T, axis=0)
    elif arr.ndim == 3:
        # (Epochs, Channels, Time) -> Transpose to (Epochs, Time, Channels)
        return np.transpose(arr, (0, 2, 1))
    elif arr.ndim == 4:
        # (Epochs, 1, Channels, Time) -> Remove dim 1 and Transpose to (Epochs, Time, Channels)
        if arr.shape[1] == 1:
            arr = np.squeeze(arr, axis=1)
        elif arr.shape[-1] == 1:
            arr = np.squeeze(arr, axis=-1)
        return np.transpose(arr, (0, 2, 1))
    else:
        raise ValueError(f"Unexpected array dimension: {arr.ndim} (shape: {arr.shape})")


def run_subject_level_mc_cv_inception(X_healthy, X_pd, SEED=42):
    X_healthy = scale_data(X_healthy)
    X_pd = scale_data(X_pd)

    # Infer input shape from single epoch: (Timepoints, Channels)
    sample_sub = X_healthy[0]
    sample_epoch = sample_sub[0] if sample_sub.ndim == 3 else sample_sub

    if sample_epoch.ndim == 2:
        # (Channels, Time) -> (Time, Channels)
        input_shape = (sample_epoch.shape[1], sample_epoch.shape[0])
    elif sample_epoch.ndim == 3:
        if sample_epoch.shape[0] == 1:  # (1, Channels, Time)
            input_shape = (sample_epoch.shape[2], sample_epoch.shape[1])
        else:  # (Channels, Time, 1)
            input_shape = (sample_epoch.shape[1], sample_epoch.shape[0])
    else:
        raise ValueError(f"Unexpected epoch shape: {sample_epoch.shape}")

    print(f"--> Inferred InceptionTime Input Shape (Timepoints, Channels): {input_shape}")

    n_hc, n_pd = len(X_healthy), len(X_pd)
    outer_kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
    thresholds = list(range(65, 95, 5))

    hc_splits = list(outer_kf.split(np.arange(n_hc)))
    pd_splits = list(outer_kf.split(np.arange(n_pd)))

    total_correct = 0
    total_subjects = 0
    fold_summary_records = []

    # InceptionTime Hyperparameter Grid Search
    param_grid = {
        'lr': [1e-3],
        'batch_size': [32],
        'depth': [6],
        'nb_filters': [32]
    }

    keys = param_grid.keys()
    all_combinations = [dict(zip(keys, combo)) for combo in product(*param_grid.values())]

    for fold in range(5):
        print(f"\n========================================")
        print(f"========== OUTER FOLD {fold+1} / 5 ==========")
        print(f"========================================")

        hc_train_all, hc_test = hc_splits[fold]
        pd_train_all, pd_test = pd_splits[fold]

        best_score = -1.0
        best_params = None
        best_threshold = 75

        hc_inner_splits = list(KFold(n_splits=3, shuffle=True, random_state=SEED).split(hc_train_all))
        pd_inner_splits = list(KFold(n_splits=3, shuffle=True, random_state=SEED).split(pd_train_all))

        for params in all_combinations:
            inner_fold_accuracies = []
            inner_fold_thresholds = []

            for inner_fold in range(3):
                hc_tr_in_idx, hc_val_in_idx = hc_inner_splits[inner_fold]
                pd_tr_in_idx, pd_val_in_idx = pd_inner_splits[inner_fold]

                hc_train_sub = [X_healthy[hc_train_all[i]] for i in hc_tr_in_idx]
                pd_train_sub = [X_pd[pd_train_all[i]] for i in pd_tr_in_idx]
                hc_val_sub = [X_healthy[hc_train_all[i]] for i in hc_val_in_idx]
                pd_val_sub = [X_pd[pd_train_all[i]] for i in pd_val_in_idx]

                # Balance classes for inner training
                X_tr_hc_bal, X_tr_pd_bal = balance_matrices_subject_wise(hc_train_sub, pd_train_sub)
                X_inner_train = np.concatenate([X_tr_hc_bal, X_tr_pd_bal], axis=0)
                y_inner_train = np.concatenate([np.zeros(len(X_tr_hc_bal)), np.ones(len(X_tr_pd_bal))], axis=0)

                # Format to 3D (Batch, Timepoints, Channels)
                X_inner_train = format_eeg_tensor_inception(X_inner_train)

                # Shuffle training data
                shuffle_idx = np.random.RandomState(SEED).permutation(len(X_inner_train))
                X_inner_train = X_inner_train[shuffle_idx]
                y_inner_train = y_inner_train[shuffle_idx]

                # Train/Val split
                val_size = int(len(X_inner_train) * 0.1)
                X_tr, y_tr = X_inner_train[val_size:], y_inner_train[val_size:]
                X_va, y_va = X_inner_train[:val_size], y_inner_train[:val_size]

                # Instantiate InceptionTime Model
                inner_model = create_inception_time(
                    input_shape=input_shape,
                    nb_classes=1,
                    depth=params['depth'],
                    nb_filters=params['nb_filters']
                )
                inner_model.compile(
                    optimizer=tf.keras.optimizers.Adam(learning_rate=params['lr']),
                    loss='binary_crossentropy',
                    metrics=['accuracy']
                )

                early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
                inner_model.fit(
                    X_tr, y_tr,
                    epochs=40, batch_size=params['batch_size'],
                    verbose=1, validation_data=(X_va, y_va), callbacks=[early_stop]
                )

                # Inner validation threshold tuning
                val_subjects = hc_val_sub + pd_val_sub
                val_labels = [0] * len(hc_val_sub) + [1] * len(pd_val_sub)

                val_subject_ratios = []
                valid_val_labels = []

                for sub, true_lbl in zip(val_subjects, val_labels):
                    sub_array = format_eeg_tensor_inception(sub)

                    if sub_array.shape[0] == 0:
                        continue

                    epoch_probs = inner_model.predict(sub_array, batch_size=params['batch_size'], verbose=0).flatten()
                    pct_pd = float(np.mean(epoch_probs) * 100)
                    val_subject_ratios.append(pct_pd)
                    valid_val_labels.append(true_lbl)

                best_t_inner, max_inner_acc = 75, -1.0
                for t in thresholds:
                    t_preds = [1 if ratio >= t else 0 for ratio in val_subject_ratios]
                    acc = accuracy_score(valid_val_labels, t_preds) if len(valid_val_labels) > 0 else 0.0
                    if acc > max_inner_acc:
                        max_inner_acc = acc
                        best_t_inner = t

                inner_fold_accuracies.append(max_inner_acc)
                inner_fold_thresholds.append(best_t_inner)

            mean_inner_acc = np.mean(inner_fold_accuracies)
            if mean_inner_acc > best_score:
                best_score = mean_inner_acc
                best_params = params
                best_threshold = int(np.median(inner_fold_thresholds))

        print(f">> Best Grid Parameters Selected: {best_params} | Threshold: {best_threshold}% (Inner Acc: {best_score:.4f})")

        # --- OUTER TRAINING & TESTING ---
        hc_train_final = [X_healthy[i] for i in hc_train_all]
        pd_train_final = [X_pd[i] for i in pd_train_all]

        X_tr_hc_final, X_tr_pd_final = balance_matrices_subject_wise(hc_train_final, pd_train_final)
        X_train_final = np.concatenate([X_tr_hc_final, X_tr_pd_final], axis=0)
        y_train_final = np.concatenate([np.zeros(len(X_tr_hc_final)), np.ones(len(X_tr_pd_final))], axis=0)

        X_train_final = format_eeg_tensor_inception(X_train_final)

        shuffle_idx_final = np.random.RandomState(SEED).permutation(len(X_train_final))
        X_train_final = X_train_final[shuffle_idx_final]
        y_train_final = y_train_final[shuffle_idx_final]

        val_size_final = int(len(X_train_final) * 0.1)
        X_tr_f, y_tr_f = X_train_final[val_size_final:], y_train_final[val_size_final:]
        X_va_f, y_va_f = X_train_final[:val_size_final], y_train_final[:val_size_final]

        final_model = create_inception_time(
            input_shape=input_shape,
            nb_classes=1,
            depth=best_params['depth'],
            nb_filters=best_params['nb_filters']
        )
        final_model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=best_params['lr']),
            loss='binary_crossentropy',
            metrics=['accuracy']
        )

        early_stop_final = callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
        final_model.fit(
            X_tr_f, y_tr_f,
            epochs=80, batch_size=best_params['batch_size'],
            verbose=1, validation_data=(X_va_f, y_va_f), callbacks=[early_stop_final]
        )

        test_subjects = [X_healthy[i] for i in hc_test] + [X_pd[i] for i in pd_test]
        test_labels = [0] * len(hc_test) + [1] * len(pd_test)
        n_hc_test = len(hc_test)
        n_pd_test = len(pd_test)

        hc_correct_count = 0
        pd_correct_count = 0

        for sub, true_label in zip(test_subjects, test_labels):
            sub_array = format_eeg_tensor_inception(sub)

            if sub_array.shape[0] == 0:
                continue

            pct_pd = float(np.mean(final_model.predict(sub_array, batch_size=best_params['batch_size'], verbose=0).flatten()) * 100)

            vote_thresholds = [best_threshold - 5, best_threshold, best_threshold + 5]
            votes = [1 if pct_pd >= t else 0 for t in vote_thresholds]
            pred = 1 if sum(votes) >= 2 else 0

            if pred == true_label:
                if true_label == 0:
                    hc_correct_count += 1
                else:
                    pd_correct_count += 1

        fold_total_correct = hc_correct_count + pd_correct_count
        fold_total_subjects = len(test_subjects)

        total_correct += fold_total_correct
        total_subjects += fold_total_subjects

        fold_acc = (fold_total_correct / fold_total_subjects) * 100 if fold_total_subjects > 0 else 0.0

        fold_summary_records.append({
            'Fold Number': fold + 1,
            'Optimal Hyperparams': str(best_params),
            'Optimal Threshold (%)': best_threshold,
            'Healthy Correct': f"{hc_correct_count}/{n_hc_test}",
            'PD Correct': f"{pd_correct_count}/{n_pd_test}",
            'Fold Accuracy (%)': f"{fold_acc:.2f}%",
            'Total Correct': f"{fold_total_correct}/{fold_total_subjects}"
        })

        print(f"Outer Fold {fold+1} Stats -> Healthy: {hc_correct_count}/{n_hc_test} | PD: {pd_correct_count}/{n_pd_test} | Acc: {fold_acc:.2f}%")

    summary_df = pd.DataFrame(fold_summary_records)
    overall_acc = (total_correct / total_subjects) * 100 if total_subjects > 0 else 0.0

    print(f"\n========================================")
    print(f"Total Combined Correct: {total_correct}/{total_subjects}")
    print(f"Overall Nested Cross-Validation Accuracy: {overall_acc:.2f}%")
    print("\n--- Nested Cross-Validation Summary ---")
    print(summary_df.to_string(index=False))

    return summary_df

In [17]:
task = 'rest'
band = 'alpha'
X_hc,X_pd = get_data(task, band, rest_ids, walk_ids)
df =run_subject_level_mc_cv_inception(X_hc, X_pd, SEED=42)
print(df)

NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


/tmp/ipykernel_58/3059442721.py:33: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)
/tmp/ipykernel_58/3059442721.py:41: RuntimeWarning: The unit for channel(s) VREF has changed from V to NA.
  raw.set_channel_types(existing_mapping)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


/tmp/ipykernel_58/3059442721.py:33: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)
/tmp/ipykernel_58/3059442721.py:41: RuntimeWarning: The unit for channel(s) VREF has changed from V to NA.
  raw.set_channel_types(existing_mapping)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


...

157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 272ms/step - accuracy: 0.5983 - loss: 0.6704

2026-08-28 20:32:16.476697: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}


157/157 ━━━━━━━━━━━━━━━━━━━━ 58s 285ms/step - accuracy: 0.6185 - loss: 0.6510 - val_accuracy: 0.5189 - val_loss: 1.7787
Epoch 2/80
157/157 ━━━━━━━━━━━━━━━━━━━━ 44s 278ms/step - accuracy: 0.6843 - loss: 0.5877 - val_accuracy: 0.6108 - val_loss: 0.6962
Epoch 3/80
157/157 ━━━━━━━━━━━━━━━━━━━━ 45s 289ms/step - accuracy: 0.7123 - loss: 0.5492 - val_accuracy: 0.5946 - val_loss: 0.7251
Epoch 4/80
157/157 ━━━━━━━━━━━━━━━━━━━━ 47s 300ms/step - accuracy: 0.7419 - loss: 0.5050 - val_accuracy: 0.5928 - val_loss: 0.7793
Epoch 5/80
157/157 ━━━━━━━━━━━━━━━━━━━━ 44s 281ms/step - accuracy: 0.7712 - loss: 0.4604 - val_accuracy: 0.5838 - val_loss: 1.1032
Epoch 6/80
157/157 ━━━━━━━━━━━━━━━━━━━━ 46s 290ms/step - accuracy: 0.8044 - loss: 0.4115 - val_accuracy: 0.6342 - val_loss: 0.9745
Epoch 7/80
157/157 ━━━━━━━━━━━━━━━━━━━━ 46s 294ms/step - accuracy: 0.8422 - loss: 0.3521 - val_accuracy: 0.6252 - val_loss: 0.8051
Epoch 8/80
157/157 ━━━━━━━━━━━━━━━━━━━━ 44s 282ms/step - accuracy: 0.8808 - loss: 0.2791 - val

2026-08-28 20:59:46.734477: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}
2026-08-28 20:59:51.881283: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

Outer Fold 5 Stats -> Healthy: 5/5 | PD: 14/23 | Acc: 67.86%

Total Combined Correct: 77/144
Overall Nested Cross-Validation Accuracy: 53.47%

--- Nested Cross-Validation Summary ---
 Fold Number                                           Optimal Hyperparams  Optimal Threshold (%) Healthy Correct PD Correct Fold Accuracy (%) Total Correct
           1 {'lr': 0.001, 'batch_size': 32, 'depth': 6, 'nb_filters': 32}                     65             4/6      14/24            60.00%         18/30
           2 {'lr': 0.001, 'batch_size': 32, 'depth': 6, 'nb_filters': 32}                     65             6/6       6/23            41.38%         12/29
           3 {'lr': 0.001, 'batch_size': 32, 'depth': 6, 'nb_filters': 32}                     65             5/6      11/23            55.17%         16/29
           4 {'lr': 0.001, 'batch_size': 32, 'depth': 6, 'nb_filters': 32}                     65             4/5       8/23            42.86%         12/28
           5 {'lr': 0.001, 'batc

In [18]:
task = 'walk'
band = 'alpha'
X_hc,X_pd = get_data(task, band, rest_ids, walk_ids)
df =run_subject_level_mc_cv_inception(X_hc, X_pd, SEED=42)
print(df)

NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


/tmp/ipykernel_58/3059442721.py:33: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)
/tmp/ipykernel_58/3059442721.py:41: RuntimeWarning: The unit for channel(s) VREF has changed from V to NA.
  raw.set_channel_types(existing_mapping)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


/tmp/ipykernel_58/3059442721.py:33: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)
/tmp/ipykernel_58/3059442721.py:41: RuntimeWarning: The unit for channel(s) VREF has changed from V to NA.
  raw.set_channel_types(existing_mapping)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


...

123/123 ━━━━━━━━━━━━━━━━━━━━ 0s 302ms/step - accuracy: 0.5295 - loss: 0.7226

2026-08-29 00:02:28.663426: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}


123/123 ━━━━━━━━━━━━━━━━━━━━ 53s 319ms/step - accuracy: 0.5336 - loss: 0.7110 - val_accuracy: 0.4564 - val_loss: 0.9682
Epoch 2/80
123/123 ━━━━━━━━━━━━━━━━━━━━ 36s 292ms/step - accuracy: 0.6384 - loss: 0.6397 - val_accuracy: 0.5642 - val_loss: 0.7267
Epoch 3/80
123/123 ━━━━━━━━━━━━━━━━━━━━ 36s 295ms/step - accuracy: 0.7188 - loss: 0.5391 - val_accuracy: 0.6560 - val_loss: 0.7927
Epoch 4/80
123/123 ━━━━━━━━━━━━━━━━━━━━ 38s 309ms/step - accuracy: 0.8010 - loss: 0.4274 - val_accuracy: 0.6720 - val_loss: 0.6708
Epoch 5/80
123/123 ━━━━━━━━━━━━━━━━━━━━ 36s 289ms/step - accuracy: 0.8651 - loss: 0.3115 - val_accuracy: 0.7202 - val_loss: 0.8868
Epoch 6/80
123/123 ━━━━━━━━━━━━━━━━━━━━ 35s 288ms/step - accuracy: 0.9150 - loss: 0.2206 - val_accuracy: 0.7225 - val_loss: 0.8011
Epoch 7/80
123/123 ━━━━━━━━━━━━━━━━━━━━ 35s 285ms/step - accuracy: 0.9298 - loss: 0.1748 - val_accuracy: 0.6422 - val_loss: 1.1911
Epoch 8/80
123/123 ━━━━━━━━━━━━━━━━━━━━ 37s 298ms/step - accuracy: 0.9410 - loss: 0.1475 - val

2026-08-29 00:17:50.126909: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}
2026-08-29 00:17:55.293043: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

Outer Fold 5 Stats -> Healthy: 3/4 | PD: 13/22 | Acc: 61.54%

Total Combined Correct: 77/133
Overall Nested Cross-Validation Accuracy: 57.89%

--- Nested Cross-Validation Summary ---
 Fold Number                                           Optimal Hyperparams  Optimal Threshold (%) Healthy Correct PD Correct Fold Accuracy (%) Total Correct
           1 {'lr': 0.001, 'batch_size': 32, 'depth': 6, 'nb_filters': 32}                     65             4/5      14/22            66.67%         18/27
           2 {'lr': 0.001, 'batch_size': 32, 'depth': 6, 'nb_filters': 32}                     65             3/5      17/22            74.07%         20/27
           3 {'lr': 0.001, 'batch_size': 32, 'depth': 6, 'nb_filters': 32}                     65             1/5      14/22            55.56%         15/27
           4 {'lr': 0.001, 'batch_size': 32, 'depth': 6, 'nb_filters': 32}                     65             4/4       4/22            30.77%          8/26
           5 {'lr': 0.001, 'batc